# Decompression Experiment — station-ecs-lab
Vertical slice: subjective knowledge vs objective world in a deterministic station simulation.

## 1. Install / import validation
On Kaggle: `!pip install tcod-ecs==5.5.0 pandas`, then make `src/` importable (clone the repo and `pip install -e .`, or add it to `sys.path` as below).

In [ ]:
import sys
sys.path.insert(0, '../src')  # repo checkout; not needed if the package is installed
import tcod.ecs, pandas as pd
from station_sim.scenarios.decompression import build_station, incident_tick, BREACH_TICK
from station_sim.simulation import simulation_tick, run
from station_sim.ecs.queries import rooms, doors, crew_members, name_of, room_named
from station_sim.ecs.helpers import adjacent_rooms, location_of
from station_sim.systems.atmosphere import trigger_breach
from station_sim.systems.logging import event_log_entries
from station_sim.experiments.runner import run_decompression_experiment
from station_sim.experiments.comparison import compare_runs, crew_df, rooms_df, facts_df, actions_df
print('imports OK')

## 2-3. World construction and station overview

In [ ]:
world = build_station(seed=82741)
for room in sorted(rooms(world), key=name_of):
    print(f'{name_of(room):15s} adjacent: {sorted(name_of(r) for r in adjacent_rooms(room))}')
print()
for crew in sorted(crew_members(world), key=name_of):
    print(f'{name_of(crew):8s} starts in {name_of(location_of(crew))}')

## 4-5. Breach injection and several simulation ticks

In [ ]:
from station_sim.domain.components import Atmosphere
trigger_breach(world, room_named(world, 'Maintenance'), leak_rate=8.0)
run(world, 15)
for room in sorted(rooms(world), key=name_of):
    print(f"{name_of(room):15s} {room.components[Atmosphere].pressure_kpa:8.2f} kPa")

## 6. Omniscient event log (ground truth — never crew knowledge)

In [ ]:
for entry in event_log_entries(world)[-25:]:
    print(entry)

## 7-9. Run the packaged experiment; pressure, crew, and knowledge tables

In [ ]:
result = run_decompression_experiment(seed=82741, comms_enabled=True, ticks=40)
display(rooms_df(result))
display(crew_df(result))
display(facts_df(result).head(15))

## 10-12. Comms-enabled vs comms-disrupted

In [ ]:
on = run_decompression_experiment(seed=82741, comms_enabled=True, ticks=40)
off = run_decompression_experiment(seed=82741, comms_enabled=False, ticks=40)
display(compare_runs(on, off))
print('--- crew, comms ON ---'); display(crew_df(on))
print('--- crew, comms OFF ---'); display(crew_df(off))
print('--- actions by kind, comms ON ---')
display(actions_df(on).groupby(['kind','status']).size().unstack(fill_value=0))

## 13. Paired multi-seed sweep
Every seed runs comms ON and comms OFF against identical initial conditions, so differences are attributable to communication. The seed varies crew personality jitter and breach severity; each seed is exactly reproducible.

In [ ]:
from station_sim.experiments.comparison import run_paired_sweep
sweep = run_paired_sweep(range(10), ticks=40)
cols = ['seed','comms_enabled','crew_injured','crew_aware_of_incident',
        'number_of_radio_messages','first_broadcast_tick','first_remote_awareness_tick',
        'incident_contained_tick','maintenance_min_pressure']
display(sweep[cols])
print('\nMeans by comms mode:')
display(sweep.groupby('comms_enabled')[['crew_injured','crew_aware_of_incident',
        'number_of_radio_messages','maintenance_min_pressure']].mean())

## 14. Knowledge propagation timeline
Who knows about the incident, from which tick, and how they learned.

In [ ]:
from station_sim.experiments.comparison import knowledge_timeline, knowledge_timeline_grid
display(knowledge_timeline(on))
grid = knowledge_timeline_grid(on)
# show every tick from first awareness onward
display(grid.loc[10:22])